# Test lc_wrapper Extension

This notebook tests that the lc_wrapper is properly installed and enabled.
- Verify that log_history metadata is recorded

## Parameters

In [ ]:
# Default parameters (will be overridden by Papermill)
notebook7_url = "http://localhost:8888/tree"
jupyter_api_url = "http://localhost:8888/api"
jupyter_token = "test-token"
default_result_path = None
close_on_fail = False
transition_timeout = 30000

In [ ]:
import tempfile

work_dir = tempfile.mkdtemp()
if default_result_path is None:
    default_result_path = work_dir
print(f"Created work directory: {work_dir}")

In [ ]:
import importlib
import re

import scripts.playwright
importlib.reload(scripts.playwright)

import scripts.notebook7
importlib.reload(scripts.notebook7)

from scripts.playwright import *
from scripts.notebook7 import *

await init_pw_context(close_on_fail=close_on_fail, last_path=default_result_path)

## Open Jupyter Notebook and wait for it to load

In [ ]:
async def _step_wait_for_loading(page):
    await page.goto(f"{notebook7_url}?token={jupyter_token}")

    # Wait for Notebook 7 file browser to load
    await expect(page.locator('.jp-DirListing')).to_be_visible(timeout=transition_timeout)

await run_pw(_step_wait_for_loading)

## Create a new notebook

In [ ]:
async def _step_create_notebook(page):
    # Create a new notebook using the notebook7 helper
    new_page = await create_new_notebook(page, kernel="Python 3", timeout=transition_timeout)
    print("✓ New notebook created")
    return new_page

await run_pw(_step_create_notebook)

## Execute a cell to trigger lc_wrapper

In [ ]:
async def _step_execute_cell(page):
    # Create and execute a cell
    await set_cell(page, 0, "code", "print('Test lc_wrapper')", timeout=transition_timeout)
    await run_cell(page, 0, True, timeout=transition_timeout)
    print("✓ Cell executed")
    
    # Save the notebook (Ctrl+S)
    await page.keyboard.press('ControlOrMeta+S')
    await page.wait_for_timeout(1000)
    print("✓ Notebook saved")

await run_pw(_step_execute_cell)

## Verify that log_history metadata is recorded

In [ ]:
async def _step_verify_meta_data(page):
    # Get the notebook path from the URL
    url = page.url
    match = re.search(r'/notebooks/(.+\.ipynb)', url)
    if not match:
        raise Exception(f"Could not extract notebook path from URL: {url}")
    
    notebook_path = match.group(1)
    print(f"Notebook path: {notebook_path}")
    
    # Fetch notebook content via Jupyter API
    api_url = f"{jupyter_api_url}/contents/{notebook_path}?token={jupyter_token}"
    response = await page.request.get(api_url)
    print(f"✓ API response status: {response.status}")
    
    notebook_data = await response.json()
    
    # Check cell meme
    cells = notebook_data['content']['cells']
    assert len(cells) > 0, "No cells in notebook"
    
    first_cell = cells[0]
    assert 'metadata' in first_cell, "No metadata in first cell"
    assert 'lc_wrapper' in first_cell['metadata'], "No lc_wrapper in first cell metadata"
    
    lc_wrapper_meta_data = first_cell['metadata']['lc_wrapper']
    assert 'log_history' in lc_wrapper_meta_data, "No 'log_history' in lc_wrapper metadata"
    assert len(lc_wrapper_meta_data['log_history']) == 1, f"Expected 1 log entry, got {len(lc_wrapper_meta_data['log_history'])}"
    
    first_log_path = lc_wrapper_meta_data['log_history'][0]
    
    print(f"Log path recorded: {first_log_path}")
    print("✓ First cell 'log_history' is a valid.")

await run_pw(_step_verify_meta_data)

## Cleanup

In [ ]:
await finish_pw_context()
!rm -rf {work_dir}